In [1]:
import gensim.downloader as api

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import numpy as np
import torch
from torch import nn, Tensor
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

2025-06-13 07:02:36.161845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749819756.264106    5671 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749819756.287673    5671 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749819756.491532    5671 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749819756.491559    5671 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749819756.491561    5671 computation_placer.cc:177] computation placer alr

In [2]:
dataset = api.load("text8")

In [3]:
VOCAB_SIZE = 20_000

tokenizer = Tokenizer(num_words = VOCAB_SIZE)
tokenizer.fit_on_texts(dataset)

sequences = tokenizer.texts_to_sequences(dataset)

In [4]:
class CBOW(nn.Module):
    def __init__(self, vocab_size: int, context_size: int, embedding_dim: int):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size, bias = False)

    def forward(self, x):
        
        emb = self.embedding(x)
        avg = emb.mean(dim = 1)
        return self.linear(avg)

In [5]:
class CustomDataset(Dataset):
    def __init__(self, sequences: list[list], context_size: int):
        super().__init__()

        self.sequences = sequences
        self.context_size = context_size
        self.half_context_size = context_size // 2

        self.samples = []
        for seq_idx, seq in enumerate(sequences):
            for j in range(self.half_context_size, len(seq) - self.half_context_size):
                self.samples.append((seq_idx, j))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[Tensor]:
        seq_idx, j = self.samples[idx]
        seq = self.sequences[seq_idx]

        #Tokens around center point
        x1 = seq[j - self.half_context_size: j]
        x2 = seq[j + 1: j + 1 + self.half_context_size]
        context = x1 + x2

        X = torch.tensor(context, dtype = torch.long)
        Y = torch.tensor(seq[j], dtype = torch.long)
        return X,Y

In [6]:
DEVICE = "cuda"

context_size = 10
embedding_dim = 50

cbow_model = CBOW(VOCAB_SIZE, context_size, embedding_dim)
optim = Adam(cbow_model.parameters(), lr = 0.0005)
criterion = nn.CrossEntropyLoss()

cbow_model.to(DEVICE)
criterion.to(DEVICE)

CrossEntropyLoss()

In [7]:
dataset = CustomDataset(sequences, context_size)
dataloader = DataLoader(dataset, batch_size = 4096, shuffle = True, pin_memory = True)

In [ ]:
from tqdm import tqdm

losses = []
cbow_model.train()

for epoch in range(50):
    e_loss = 0

    pbar = tqdm(dataloader, total = len(dataloader))
    for (X, Y) in pbar:
        X = X.to(DEVICE)
        Y = Y.to(DEVICE)

        optim.zero_grad()

        output = cbow_model(X)
        loss = criterion(output, Y)
        
        loss.backward()
        optim.step()

        e_loss += loss.item()

        pbar.set_description(f"Epoch: {epoch},  Loss: {loss.item():0.4f}")

    avg_loss = e_loss / len(dataloader)
    losses.append(avg_loss)
    pbar.set_description(f"Epoch: {epoch},  Loss: {avg_loss:0.4f}")


Epoch: 49,  Loss: 6.0041: 100%|██████████| 3905/3905 [06:12<00:00, 10.49it/s]


In [25]:
embeddings = cbow_model.state_dict()["linear.weight"].to("cpu")

In [43]:
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors = 5, algorithm = "ball_tree")
neighbors.fit(embeddings)

NearestNeighbors(algorithm='ball_tree')

In [31]:
queen_idx = tokenizer.word_index["queen"]
queen = embeddings[queen_idx:queen_idx+1]

distances, indices = neighbors.kneighbors(queen)
indices

array([[ 903,  187, 1061,  388,  484]])

In [32]:
for idx in indices[0]:
    print(tokenizer.index_word[idx])

queen
king
prince
son
head


In [35]:
def print_neighbors(query: str):

    query_idx = tokenizer.word_index[query]
    query_emb = embeddings[query_idx:query_idx + 1]

    distances, indices = neighbors.kneighbors(query_emb)

    for idx in indices[0]:
        print(tokenizer.index_word[idx])



In [36]:
print_neighbors("uncle")

uncle
sons
wife
brother
mother


In [37]:
print_neighbors("paris")

paris
de
le
la
moscow


In [38]:
print_neighbors("japan")

japan
china
europe
india
southern


In [39]:
print_neighbors("election")

election
candidate
elections
vote
parliament


In [40]:
print_neighbors("california")

california
texas
home
southern
lake
